# 03 · Rules across columns, and findings

What this notebook does:
- checks rules that involve more than one column: dates, price vs sale code, multi-parcel deeds, duplicates, code descriptions
- answers the questions the silver and gold design depends on (§4.7)
- lists every finding and open question from notebooks 01–03 in one place (§5)

In [1]:
from helpers import connect, show, profile, by_year, values_by_year, DATA_DIR

con, years = connect()
print(f"years: {years[0]}-{years[-1]} ({len(years)} files)")
show("SELECT count(*) AS total_rows FROM s")

years: 2012-2026 (15 files)
┌────────────┐
│ total_rows │
│   int64    │
├────────────┤
│     506977 │
└────────────┘



## 4. Rules across columns

### 4.1 Dates
**Questions:**
- Read as year-month-day or as year-day-month: which reading keeps the sale date on or before the record date?
- Does each file contain only its own year?
- Can a sale be recorded *before* it happened? How long does recording usually take?
- Are the most recent months complete?

In [2]:
# The same rows read both ways: the correct reading should rarely put the sale after the record
show("""
SELECT count(*) FILTER (WHERE trim(RECORDDATE) <> '') AS rows_with_record_date,
       count(*) FILTER (WHERE try_strptime(SALEDATE, '%Y-%m-%d') > try_strptime(RECORDDATE, '%Y-%m-%d')) AS sale_after_record_if_y_m_d,
       count(*) FILTER (WHERE try_strptime(SALEDATE, '%Y-%d-%m') > try_strptime(RECORDDATE, '%Y-%d-%m')) AS sale_after_record_if_y_d_m,
       count(*) FILTER (WHERE trim(RECORDDATE) <> '' AND (try_strptime(SALEDATE, '%Y-%d-%m') IS NULL
                                                          OR try_strptime(RECORDDATE, '%Y-%d-%m') IS NULL)) AS not_a_date_if_y_d_m
FROM s
""")

┌───────────────────────┬────────────────────────────┬────────────────────────────┬─────────────────────┐
│ rows_with_record_date │ sale_after_record_if_y_m_d │ sale_after_record_if_y_d_m │ not_a_date_if_y_d_m │
│         int64         │           int64            │           int64            │        int64        │
├───────────────────────┼────────────────────────────┼────────────────────────────┼─────────────────────┤
│                505728 │                        733 │                       7653 │              398182 │
└───────────────────────┴────────────────────────────┴────────────────────────────┴─────────────────────┘



In [3]:
show("""
SELECT file_year,
       count(*) FILTER (WHERE year(try_strptime(SALEDATE, '%Y-%m-%d')) <> file_year) AS sale_year_mismatch,
       count(*) FILTER (WHERE try_strptime(SALEDATE, '%Y-%m-%d') IS NULL)             AS sale_date_unparsed,
       count(*) FILTER (WHERE trim(RECORDDATE) = '')                                  AS record_blank
FROM s GROUP BY 1 ORDER BY 1
""")

┌───────────┬────────────────────┬────────────────────┬──────────────┐
│ file_year │ sale_year_mismatch │ sale_date_unparsed │ record_blank │
│   int32   │       int64        │       int64        │    int64     │
├───────────┼────────────────────┼────────────────────┼──────────────┤
│      2012 │                  0 │                  0 │          190 │
│      2013 │                  0 │                  0 │          332 │
│      2014 │                  0 │                  0 │          327 │
│      2015 │                  0 │                  0 │          215 │
│      2016 │                  0 │                  0 │          150 │
│      2017 │                  0 │                  0 │           16 │
│      2018 │                  0 │                  0 │           12 │
│      2019 │                  0 │                  0 │            1 │
│      2020 │                  0 │                  0 │            1 │
│      2021 │                  0 │                  0 │            3 │
│     

In [4]:
show("""
WITH d AS (SELECT try_strptime(SALEDATE, '%Y-%m-%d')::DATE AS sale_date,
                  try_strptime(nullif(trim(RECORDDATE), ''), '%Y-%m-%d')::DATE AS record_date FROM s)
SELECT count(*) FILTER (WHERE sale_date IS NULL)        AS sale_unparsed,
       count(*) FILTER (WHERE record_date IS NULL)      AS record_blank_or_unparsed,
       count(*) FILTER (WHERE sale_date > record_date)  AS sale_after_record,
       quantile_cont(record_date - sale_date, 0.5)      AS lag_days_p50,
       quantile_cont(record_date - sale_date, 0.9)      AS lag_days_p90,
       quantile_cont(record_date - sale_date, 0.99)     AS lag_days_p99,
       max(record_date - sale_date)                     AS lag_days_max
FROM d
""")

┌───────────────┬──────────────────────────┬───────────────────┬──────────────┬──────────────┬──────────────┬──────────────┐
│ sale_unparsed │ record_blank_or_unparsed │ sale_after_record │ lag_days_p50 │ lag_days_p90 │ lag_days_p99 │ lag_days_max │
│     int64     │          int64           │       int64       │    double    │    double    │    double    │    int64     │
├───────────────┼──────────────────────────┼───────────────────┼──────────────┼──────────────┼──────────────┼──────────────┤
│             0 │                     1249 │               733 │          6.0 │         33.0 │        221.0 │         5355 │
└───────────────┴──────────────────────────┴───────────────────┴──────────────┴──────────────┴──────────────┴──────────────┘



In [5]:
# Sales per month for the 12 whole months up to the latest sale in the data: are the most recent months complete?
show("""
WITH d AS (SELECT try_strptime(SALEDATE, '%Y-%m-%d')::DATE AS sale_date FROM s),
     last AS (SELECT max(sale_date) AS last_sale FROM d)
SELECT strftime(sale_date, '%Y-%m') AS month, count(*) AS sales
FROM d, last WHERE sale_date >= date_trunc('month', last_sale) - INTERVAL 12 MONTH
GROUP BY 1 ORDER BY 1
""")

┌─────────┬───────┐
│  month  │ sales │
│ varchar │ int64 │
├─────────┼───────┤
│ 2025-10 │  2874 │
│ 2025-11 │  2438 │
│ 2025-12 │  2783 │
│ 2026-01 │  1901 │
│ 2026-02 │  1948 │
│ 2026-03 │  2526 │
│ 2026-04 │  2547 │
│ 2026-05 │  2751 │
│ 2026-06 │  3030 │
│ 2026-07 │  2976 │
│ 2026-08 │  2608 │
│ 2026-09 │  2222 │
│ 2026-10 │   973 │
└─────────┴───────┘
      13 rows    



**What this shows:** read as year-month-day, 733 rows have the sale after the record. Read as year-day-month, 7,653 rows do, and 398,182 values wouldn't even be dates. So year-month-day is confirmed. No row has a sale year different from its file year.

Recording usually follows the sale quickly: half within 6 days, 90% within 33 days, 99% within 221 days, but the longest is 5,355 days. The 733 rows with the sale after the record are an open question.

The last query counts rows per sale month (its column is called `sales`, but it counts every record, not only valid sales). 2026-10 has only 973 rows and 2026-09 has 2,222, fewer than 2026-06 (3,030) and 2026-07 (2,976). The latest change to the 2026 file added 289 rows and removed 169, while its last sale date stayed 2026-10-22, so the new rows are older sales recorded late. So the last month or two will keep growing and can't be compared with complete months yet.

**Open questions:** are sales after their record date typing errors, and which date is right? How late can a sale still be added, and how should the most recent months be shown?

### 4.2 Price
**Questions:**
- How many prices are $0, $1 or blank, and which sale codes do they come from?
- What do the extremes look like?

In [6]:
show("""
SELECT CASE WHEN trim(PRICE) = '' THEN 'blank'
            WHEN p IS NULL THEN 'not a whole number'
            WHEN p = 0 THEN '$0'
            WHEN p = 1 THEN '$1'
            WHEN p < 1000 THEN '$2-999'
            WHEN p < 100000 THEN '$1k-100k'
            WHEN p < 1000000 THEN '$100k-1M'
            ELSE '$1M+' END AS price_band,
       count(*) AS rows
FROM (SELECT PRICE, try_cast(PRICE AS BIGINT) AS p FROM s) GROUP BY 1 ORDER BY 2 DESC
""")

┌────────────┬────────┐
│ price_band │  rows  │
│  varchar   │ int64  │
├────────────┼────────┤
│ $100k-1M   │ 237827 │
│ $1k-100k   │ 124912 │
│ $1         │ 105558 │
│ $0         │  15813 │
│ $2-999     │  11008 │
│ $1M+       │   8751 │
│ blank      │   3108 │
└────────────┴────────┘



In [7]:
show("""
SELECT SALECODE, SALEDESC, count(*) AS rows,
       count(*) FILTER (WHERE try_cast(PRICE AS BIGINT) <= 1) AS price_le_1,
       median(try_cast(PRICE AS BIGINT)) AS median_price
FROM s GROUP BY 1, 2 ORDER BY rows DESC
""", rows=50)

┌──────────┬─────────────────────────────────────────┬────────┬────────────┬──────────────┐
│ SALECODE │                SALEDESC                 │  rows  │ price_le_1 │ median_price │
│ varchar  │                 varchar                 │ int64  │   int64    │    double    │
├──────────┼─────────────────────────────────────────┼────────┼────────────┼──────────────┤
│ 3        │ LOVE AND AFFECTION SALE                 │ 104703 │      96834 │          1.0 │
│ 0        │ VALID SALE                              │ 100458 │         12 │     209900.0 │
│ H        │ MULTI-PARCEL SALE                       │  67492 │      20293 │      64000.0 │
│ 14       │ TIME ON MARKET (INSUFF/EXCESS)          │  45338 │         45 │     105000.0 │
│ 36       │ QUIT CLAIM                              │  40189 │        156 │     134900.0 │
│ 9        │ OTHER INVALID SALES INDICATED           │  30928 │        956 │     194657.5 │
│ 99       │ CORRECTIVE DEED / DUPLICATE SALE        │  16769 │        493 │    

In [8]:
show("SELECT PRICE, SALECODE, SALEDESC, MUNIDESC, FULL_ADDRESS FROM s ORDER BY try_cast(PRICE AS BIGINT) DESC NULLS LAST LIMIT 10")

┌───────────┬──────────┬──────────────────────────────────┬────────────────────────┬──────────────────────────────────────────────┐
│   PRICE   │ SALECODE │             SALEDESC             │        MUNIDESC        │                 FULL_ADDRESS                 │
│  varchar  │ varchar  │             varchar              │        varchar         │                   varchar                    │
├───────────┼──────────┼──────────────────────────────────┼────────────────────────┼──────────────────────────────────────────────┤
│ 148752900 │ 36       │ QUIT CLAIM                       │ 1st Ward  - PITTSBURGH │ 301 GRANT ST, PITTSBURGH, PA 15219           │
│ 124400000 │ 0        │ VALID SALE                       │ 2nd Ward - PITTSBURGH  │ 1001 LIBERTY AVE, PITTSBURGH, PA 15222       │
│ 82000000  │ H        │ MULTI-PARCEL SALE                │ 22nd Ward - PITTSBURGH │ 1205-1231 FEDERAL ST, PITTSBURGH, PA 15212   │
│ 82000000  │ H        │ MULTI-PARCEL SALE                │ 22nd Ward - PITT

**What this shows:** most rows aren't market prices: 105,558 rows are $1, 15,813 are $0, 11,008 are $2 to $999 and 3,108 are blank. LOVE AND AFFECTION SALE has 96,834 of its 104,703 rows at $1 or less (median 1), while VALID SALE has only 12 (median 209,900). MULTI-PARCEL SALE has 20,293 rows at $1 or less, and GOVERNMENT SALE has a median of 2,190.

The biggest prices repeat across parcels of the same deed, so adding up PRICE over rows counts the same money more than once. So which rows count as a sale has to come from the sale code, not the price.

**Open question:** which sale codes count as a real market sale?

### 4.3 Deeds with several parcels
**Question:** when one deed covers several parcels, what's in PRICE on each row: a share of the total, or the total repeated?

In [9]:
# Placeholder = book OR page blank/zeros, the same rule as 01 §2.4
show("""
SELECT DEEDBOOK, DEEDPAGE, PARID, PRICE, SALECODE, SALEDESC
FROM s
WHERE (DEEDBOOK, DEEDPAGE) IN (SELECT (DEEDBOOK, DEEDPAGE) FROM s WHERE trim(DEEDBOOK, '0 ') <> '' AND trim(DEEDPAGE, '0 ') <> ''
                               GROUP BY DEEDBOOK, DEEDPAGE HAVING count(DISTINCT PARID) BETWEEN 3 AND 4
                               ORDER BY DEEDBOOK, DEEDPAGE LIMIT 3)
ORDER BY 1, 2, 3
""")

┌──────────┬──────────┬──────────────────┬─────────┬──────────┬───────────────────────────────┐
│ DEEDBOOK │ DEEDPAGE │      PARID       │  PRICE  │ SALECODE │           SALEDESC            │
│ varchar  │ varchar  │     varchar      │ varchar │ varchar  │            varchar            │
├──────────┼──────────┼──────────────────┼─────────┼──────────┼───────────────────────────────┤
│ 14780    │ 273      │ 1084A00111000000 │ 1       │ 9        │ OTHER INVALID SALES INDICATED │
│ 14780    │ 273      │ 1084A00189000000 │ 1       │ 3        │ LOVE AND AFFECTION SALE       │
│ 14780    │ 273      │ 1084A00194000000 │ 1       │ 3        │ LOVE AND AFFECTION SALE       │
│ 14781    │ 266      │ 0446E00099000000 │ 1       │ 3        │ LOVE AND AFFECTION SALE       │
│ 14781    │ 266      │ 0446K00288000000 │ 1       │ 3        │ LOVE AND AFFECTION SALE       │
│ 14781    │ 266      │ 0447M00364000000 │ 1       │ 3        │ LOVE AND AFFECTION SALE       │
│ 14782    │ 228      │ 0154S00233000000

In [10]:
show("""
SELECT count(*) AS multi_parcel_deeds,
       count(*) FILTER (WHERE distinct_prices = 1) AS same_price_on_every_parcel
FROM (SELECT DEEDBOOK, DEEDPAGE, count(DISTINCT PRICE) AS distinct_prices
      FROM s WHERE trim(DEEDBOOK, '0 ') <> '' AND trim(DEEDPAGE, '0 ') <> ''
      GROUP BY DEEDBOOK, DEEDPAGE HAVING count(DISTINCT PARID) > 1)
""")

┌────────────────────┬────────────────────────────┐
│ multi_parcel_deeds │ same_price_on_every_parcel │
│       int64        │           int64            │
├────────────────────┼────────────────────────────┤
│              25170 │                      24408 │
└────────────────────┴────────────────────────────┘



**Question:** where the parcels of one deed have different prices, what kind of deed is it?

In [11]:
# A deed whose parcels carry different instrument types is shown as (mixed) instead of one arbitrary type
show("""
SELECT INSTRTYP, INSTRTYPDESC, count(*) AS multi_parcel_deeds,
       count(*) FILTER (WHERE distinct_prices > 1) AS with_different_prices
FROM (SELECT DEEDBOOK, DEEDPAGE,
             CASE WHEN count(DISTINCT INSTRTYP) > 1 THEN '(mixed)' ELSE any_value(INSTRTYP) END AS INSTRTYP,
             CASE WHEN count(DISTINCT INSTRTYP) > 1 THEN 'several instrument types' ELSE any_value(INSTRTYPDESC) END AS INSTRTYPDESC,
             count(DISTINCT PRICE) AS distinct_prices
      FROM s WHERE trim(DEEDBOOK, '0 ') <> '' AND trim(DEEDPAGE, '0 ') <> ''
      GROUP BY 1, 2 HAVING count(DISTINCT PARID) > 1)
GROUP BY 1, 2 ORDER BY with_different_prices DESC LIMIT 10
""")

┌──────────┬──────────────────────────┬────────────────────┬───────────────────────┐
│ INSTRTYP │       INSTRTYPDESC       │ multi_parcel_deeds │ with_different_prices │
│ varchar  │         varchar          │       int64        │         int64         │
├──────────┼──────────────────────────┼────────────────────┼───────────────────────┤
│ TS       │ TREASURER DEED           │                301 │                   256 │
│ DE       │ DEED                     │              14976 │                   210 │
│ (mixed)  │ several instrument types │                357 │                   112 │
│ CO       │ CORRECTIVE DEED          │                490 │                    78 │
│ SW       │ SPECIAL WARRANTY         │               5586 │                    52 │
│ QC       │ QUIT CLAIM               │                848 │                    17 │
│ GW       │ GENERAL WARRANTY         │                991 │                    10 │
│ FD       │ FIDUCIARY DEED           │                462 │     

**What this shows:** the data dictionary is right. Of 25,170 deeds that cover more than one parcel, 24,408 have the same price on every parcel, so the price is the deed's total repeated on each parcel.

### 4.4 Duplicates
**Questions:**
- Are there exact duplicate rows (ignoring `_id`) inside a file?
- Does the same parcel + deed appear in more than one year file?

In [12]:
cols = [c for c in con.sql("SELECT * FROM s LIMIT 0").columns if c not in ("_id", "file_year")]
col_list = ", ".join(f'"{c}"' for c in cols)
show(f"""
SELECT count(*) AS duplicate_groups, coalesce(sum(n - 1), 0) AS extra_rows
FROM (SELECT count(*) AS n FROM s GROUP BY file_year, {col_list} HAVING count(*) > 1)
""")

┌──────────────────┬────────────┐
│ duplicate_groups │ extra_rows │
│      int64       │   int128   │
├──────────────────┼────────────┤
│                0 │          0 │
└──────────────────┴────────────┘



In [13]:
show("""
SELECT PARID, DEEDBOOK, DEEDPAGE,
       list(file_year || ' | ' || SALEDATE || ' | $' || PRICE || ' | ' || SALECODE ORDER BY file_year) AS appearances
FROM s WHERE trim(DEEDBOOK, '0 ') <> '' AND trim(DEEDPAGE, '0 ') <> ''
GROUP BY 1, 2, 3 HAVING count(DISTINCT file_year) > 1
""", width=300)

┌──────────────────┬──────────┬──────────┬──────────────────────────────────────────────────────────────────────┐
│      PARID       │ DEEDBOOK │ DEEDPAGE │                             appearances                              │
│     varchar      │ varchar  │ varchar  │                              varchar[]                               │
├──────────────────┼──────────┼──────────┼──────────────────────────────────────────────────────────────────────┤
│ 0419C00006000000 │ 20137    │ 285      │ [2021 | 2021-08-05 | $146250 | 13, 2025 | 2025-08-05 | $146250 | 35] │
│ 0980R00003000000 │ 15892    │ 550      │ [2015 | 2015-02-25 | $1 | 3, 2022 | 2022-02-25 | $1 | 3]             │
│ 0202C00006000000 │ 18647    │ 534      │ [2016 | 2016-10-08 | $173000 | 36, 2021 | 2021-10-08 | $173000 | 0]  │
│ 0381K00199000000 │ 15933    │ 590      │ [2013 | 2013-10-04 | $10 | 3, 2015 | 2015-03-16 | $10 | 3]           │
│ 0153J00155000000 │ 18646    │ 293      │ [2018 | 2018-10-12 | $300000 | 0, 2021 | 2021

**What this shows:** no row is an exact copy of another row in the same file (0 duplicate groups). 11 parcel + deed pairs appear in 2 different year files, and in 8 of them the month and day are the same but the year is different (e.g. 2018-10-12 and 2021-10-12 on the same deed).

*Assumptions:*
- the sale year was typed wrong in one of the two copies.
- no two files share a sale year, but one sale can still appear twice: 293 real deed references appear in more than one file.

**Open question:** for the same parcel and deed in two year files, which copy has the right sale date?

**Questions:**
- Does a parcel have valid-coded rows on the same sale date under two *different* deeds? If each deed covers one parcel, both would pass the gold rule and both prices would be used.
- Is the same parcel sold again, valid-coded, within a few months?

In [14]:
# The same parcel with valid-coded rows on one sale date under two or more different real deeds.
# Does each of those deeds pass the gold rule (one parcel, one sale date), so both prices are used?
show("""
WITH deeds AS (SELECT DEEDBOOK, DEEDPAGE, count(DISTINCT PARID) = 1 AND count(DISTINCT SALEDATE) = 1 AS single
               FROM s WHERE trim(DEEDBOOK, '0 ') <> '' AND trim(DEEDPAGE, '0 ') <> '' GROUP BY 1, 2),
     v AS (SELECT * FROM s JOIN deeds USING (DEEDBOOK, DEEDPAGE) WHERE SALECODE IN ('0', 'U', 'UR')),
     p AS (SELECT PARID, SALEDATE, count(DISTINCT (DEEDBOOK, DEEDPAGE)) AS deeds,
                  count(DISTINCT (DEEDBOOK, DEEDPAGE)) FILTER (WHERE single) AS single_deeds,
                  count(DISTINCT PRICE) AS prices, count(DISTINCT INSTRTYP) AS instruments
           FROM v GROUP BY 1, 2 HAVING count(DISTINCT (DEEDBOOK, DEEDPAGE)) > 1)
SELECT count(*) AS parcel_dates, sum(deeds) AS deeds,
       count(*) FILTER (WHERE single_deeds = deeds) AS every_deed_passes_gold_rule,
       count(*) FILTER (WHERE prices = 1) AS same_price, count(*) FILTER (WHERE prices > 1) AS different_prices,
       count(*) FILTER (WHERE instruments > 1) AS different_instruments
FROM p
""")
show("""
WITH v AS (SELECT * FROM s WHERE SALECODE IN ('0', 'U', 'UR') AND trim(DEEDBOOK, '0 ') <> '' AND trim(DEEDPAGE, '0 ') <> ''),
     p AS (SELECT PARID, SALEDATE FROM v GROUP BY 1, 2 HAVING count(DISTINCT (DEEDBOOK, DEEDPAGE)) > 1 ORDER BY 1, 2 LIMIT 5)
SELECT PARID, SALEDATE, DEEDBOOK, DEEDPAGE, PRICE, SALECODE, INSTRTYP, RECORDDATE
FROM v JOIN p USING (PARID, SALEDATE) ORDER BY 1, 2, 3, 4
""")

┌──────────────┬────────┬─────────────────────────────┬────────────┬──────────────────┬───────────────────────┐
│ parcel_dates │ deeds  │ every_deed_passes_gold_rule │ same_price │ different_prices │ different_instruments │
│    int64     │ int128 │            int64            │   int64    │      int64       │         int64         │
├──────────────┼────────┼─────────────────────────────┼────────────┼──────────────────┼───────────────────────┤
│           60 │    120 │                          59 │         46 │               14 │                    16 │
└──────────────┴────────┴─────────────────────────────┴────────────┴──────────────────┴───────────────────────┘

┌──────────────────┬────────────┬──────────┬──────────┬─────────┬──────────┬──────────┬────────────┐
│      PARID       │  SALEDATE  │ DEEDBOOK │ DEEDPAGE │  PRICE  │ SALECODE │ INSTRTYP │ RECORDDATE │
│     varchar      │  varchar   │ varchar  │ varchar  │ varchar │ varchar  │ varchar  │  varchar   │
├──────────────────┼────

In [15]:
# Valid-coded sales of the same parcel on different dates, less than 180 days apart (consecutive valid sales)
show("""
WITH v AS (SELECT DISTINCT PARID, SALEDATE::DATE AS sale_date, try_cast(PRICE AS BIGINT) AS price
           FROM s WHERE SALECODE IN ('0', 'U', 'UR') AND trim(DEEDBOOK, '0 ') <> '' AND trim(DEEDPAGE, '0 ') <> ''),
     g AS (SELECT PARID, sale_date - lag(sale_date) OVER w AS gap_days, price, lag(price) OVER w AS prev_price
           FROM v WINDOW w AS (PARTITION BY PARID ORDER BY sale_date))
SELECT count(*) AS resale_pairs, count(DISTINCT PARID) AS parcels,
       count(*) FILTER (WHERE gap_days < 30) AS under_30_days,
       quantile_disc(gap_days, 0.5) AS median_gap_days,
       count(*) FILTER (WHERE price = prev_price) AS same_price,
       count(*) FILTER (WHERE price > 1.2 * prev_price) AS price_up_over_20pct
FROM g WHERE gap_days > 0 AND gap_days < 180
""")

┌──────────────┬─────────┬───────────────┬─────────────────┬────────────┬─────────────────────┐
│ resale_pairs │ parcels │ under_30_days │ median_gap_days │ same_price │ price_up_over_20pct │
│    int64     │  int64  │     int64     │      int64      │   int64    │        int64        │
├──────────────┼─────────┼───────────────┼─────────────────┼────────────┼─────────────────────┤
│          182 │     182 │            36 │             107 │         47 │                  75 │
└──────────────┴─────────┴───────────────┴─────────────────┴────────────┴─────────────────────┘



**What this shows:** 60 parcel + sale date pairs have valid-coded rows on two different real deeds (120 deeds). In 59 of them both deeds cover one parcel on one sale date, so both prices would pass the gold rule and one sale of one parcel would be counted twice. 46 pairs have the same price on both deeds, 14 have different prices, and 16 differ in instrument type. The two deeds are usually a few pages apart in the same book and recorded the same day (e.g. `15409/192` at $185,000 and `15409/199` at $176,000).

182 pairs of valid-coded sales of the same parcel are less than 180 days apart (182 parcels, median 107 days, 36 under 30 days). 47 repeat the same price; 75 resell for more than 20% above the previous price.

**So:** the same price on neighbouring pages looks like one sale recorded twice. Different prices on the same day could be two sales back to back (A sells to B, B to C), where both are real. The data can't tell which. Until the County answers, gold uses neither price and counts both in `valid_records_excluded`. Resales a few months apart mostly look like real flips (75 with a big price rise), but a resale at the same price may be a duplicate entry. (F-32, F-33)

### 4.5 Codes and their descriptions
**Question:** does every code have exactly one description, and the other way round?

In [16]:
show("SELECT SALECODE, count(DISTINCT SALEDESC) AS descs, any_value(SALEDESC) AS example, count(*) AS rows FROM s GROUP BY 1 ORDER BY rows DESC", rows=50)

┌──────────┬───────┬─────────────────────────────────────────┬────────┐
│ SALECODE │ descs │                 example                 │  rows  │
│ varchar  │ int64 │                 varchar                 │ int64  │
├──────────┼───────┼─────────────────────────────────────────┼────────┤
│ 3        │     1 │ LOVE AND AFFECTION SALE                 │ 104703 │
│ 0        │     1 │ VALID SALE                              │ 100458 │
│ H        │     1 │ MULTI-PARCEL SALE                       │  67492 │
│ 14       │     1 │ TIME ON MARKET (INSUFF/EXCESS)          │  45338 │
│ 36       │     1 │ QUIT CLAIM                              │  40189 │
│ 9        │     1 │ OTHER INVALID SALES INDICATED           │  30928 │
│ 99       │     1 │ CORRECTIVE DEED / DUPLICATE SALE        │  16769 │
│ AA       │     1 │ SALE NOT ANALYZED                       │  16426 │
│ 1        │     1 │ SHERIFF SALE                            │  15134 │
│ UR       │     1 │ OTHER                                   │  

In [17]:
show("SELECT SALEDESC, list(DISTINCT SALECODE) AS codes FROM s GROUP BY 1 HAVING count(DISTINCT SALECODE) > 1")

┌──────────┬──────────────────────────────────────────────────────────────────────────────┐
│ SALEDESC │                                    codes                                     │
│ varchar  │                                  varchar[]                                   │
├──────────┼──────────────────────────────────────────────────────────────────────────────┤
│ OTHER    │ [XX, YY, VH, S, 28, 50, U, 29, 4, 21, ND, VB, PX, Z, UR, 5, X, UX, 8, V, 0X] │
└──────────┴──────────────────────────────────────────────────────────────────────────────┘



In [18]:
show("SELECT MUNICODE, count(DISTINCT MUNIDESC) AS descs, count(DISTINCT trim(MUNIDESC)) AS trimmed_descs FROM s GROUP BY 1 HAVING count(DISTINCT MUNIDESC) > 1")

┌──────────┬───────┬───────────────┐
│ MUNICODE │ descs │ trimmed_descs │
│ varchar  │ int64 │     int64     │
└──────────┴───────┴───────────────┘
               0 rows             



In [19]:
show("SELECT SCHOOLCODE, list(DISTINCT SCHOOLDESC) AS descs FROM s GROUP BY 1 HAVING count(DISTINCT SCHOOLDESC) > 1")

┌────────────┬───────────┐
│ SCHOOLCODE │   descs   │
│  varchar   │ varchar[] │
└────────────┴───────────┘
          0 rows        



**What this shows:** every SALECODE has exactly one description, and every municipality and school code has one name. But the description OTHER is shared by 21 different codes, including the valid codes U and UR and the discontinued codes. So the code has to be kept: the description alone loses information and can't tell valid from invalid.

### 4.6 Summary of one year
`SUMMARIZE` gives min / max / distinct values / % null for every column of one table. Change the year to look at another file.

In [20]:
show(f"SUMMARIZE bronze.sales_{years[-1]}", rows=30)

┌─────────────────────────┬─────────────┬────────────────────────────────────┬────────────────────────────────┬───────────────┬────────┬────────┬────────┬────────┬────────┬───────┬─────────────────┐
│       column_name       │ column_type │                min                 │              max               │ approx_unique │  avg   │  std   │  q25   │  q50   │  q75   │ count │ null_percentage │
│         varchar         │   varchar   │              varchar               │            varchar             │     int64     │ "null" │ "null" │ "null" │ "null" │ "null" │ int64 │  decimal(9,2)   │
├─────────────────────────┼─────────────┼────────────────────────────────────┼────────────────────────────────┼───────────────┼────────┼────────┼────────┼────────┼────────┼───────┼─────────────────┤
│ _id                     │ VARCHAR     │ 2254297                            │ 2277778                        │         24516 │ NULL   │ NULL   │ NULL   │ NULL   │ NULL   │ 23482 │            0.00 │
│ PAR

### 4.7 Questions for the silver and gold design

**Question A:** where do the valid-coded rows sit: on single-parcel deeds, on multi-parcel deeds, or on placeholder references? A row's price is the price of one parcel only when its deed covers one parcel. And do valid-coded rows have usable prices?

In [21]:
# Valid-coded rows (SALECODE 0, U, UR = "Valid Sale" in the data dictionary), by kind of deed reference
show("""
WITH r AS (SELECT *, trim(DEEDBOOK, '0 ') = '' OR trim(DEEDPAGE, '0 ') = '' AS placeholder,
                  SALECODE IN ('0', 'U', 'UR') AS valid FROM s),
     d AS (SELECT DEEDBOOK, DEEDPAGE, count(DISTINCT PARID) AS parcels, count(*) AS n,
                  count(*) FILTER (WHERE valid) AS valid_rows, count(DISTINCT SALEDATE) AS sale_dates,
                  count(DISTINCT PRICE) AS prices
           FROM r WHERE NOT placeholder GROUP BY 1, 2)
SELECT CASE WHEN parcels = 1 THEN '1 single-parcel deed' ELSE '2 multi-parcel deed' END AS deed_type,
       count(*) AS deeds,
       count(*) FILTER (WHERE valid_rows > 0) AS deeds_with_valid_row,
       count(*) FILTER (WHERE valid_rows > 0 AND valid_rows < n) AS mixed_valid_and_invalid,
       count(*) FILTER (WHERE valid_rows > 0 AND sale_dates > 1) AS valid_with_several_dates,
       count(*) FILTER (WHERE valid_rows > 0 AND prices > 1) AS valid_with_several_prices,
       sum(valid_rows) AS valid_rows
FROM d GROUP BY 1
UNION ALL SELECT '3 placeholder deed', NULL, NULL, NULL, NULL, NULL, count(*) FILTER (WHERE valid) FROM r WHERE placeholder
UNION ALL SELECT '4 all rows', NULL, NULL, NULL, NULL, NULL, count(*) FILTER (WHERE valid) FROM r
ORDER BY 1
""", width=300)

# Valid-coded rows with no price or a nominal price
show("""
SELECT SALECODE, count(*) AS rows,
       count(*) FILTER (WHERE trim(PRICE) = '') AS blank_price,
       count(*) FILTER (WHERE try_cast(PRICE AS BIGINT) <= 1) AS price_le_1,
       count(*) FILTER (WHERE try_cast(PRICE AS BIGINT) BETWEEN 2 AND 999) AS price_2_to_999
FROM s WHERE SALECODE IN ('0', 'U', 'UR') GROUP BY 1 ORDER BY 1
""")

┌──────────────────────┬────────┬──────────────────────┬─────────────────────────┬──────────────────────────┬───────────────────────────┬────────────┐
│      deed_type       │ deeds  │ deeds_with_valid_row │ mixed_valid_and_invalid │ valid_with_several_dates │ valid_with_several_prices │ valid_rows │
│       varchar        │ int64  │        int64         │          int64          │          int64           │           int64           │   int128   │
├──────────────────────┼────────┼──────────────────────┼─────────────────────────┼──────────────────────────┼───────────────────────────┼────────────┤
│ 1 single-parcel deed │ 434029 │               114440 │                      31 │                        8 │                         2 │     114440 │
│ 2 multi-parcel deed  │  25170 │                  145 │                     126 │                       72 │                        72 │        165 │
│ 3 placeholder deed   │   NULL │                 NULL │                    NULL │            

**What I found:** 114,611 rows have a valid sale code. 114,440 of them are on 114,440 single-parcel deeds, one valid row per deed (31 of those deeds also have a second, invalid row, and 8 have more than one sale date). 165 are on 145 multi-parcel deeds, of which 126 mix valid and invalid rows and 72 have several prices. 6 are on placeholder references. No valid-coded row has a blank price; 14 are $1 or less and 45 are $2 to $999.

**So:** using only single-parcel deeds for prices drops just 171 valid-coded rows (165 + 6, 0.15%) and removes every repeated deed total. Token prices among valid sales are rare, so the sale code decides what counts, not a price limit.

**Question B:** in the 90 repeated parcel + deed + sale date groups, what differs, does one row have code 99, and can both rows be valid-coded? If no group has two valid rows, keeping only valid rows never has to choose between two. The second query recounts the 88 groups from 01 §2.4, whose numbers were first read from the printed rows.

In [22]:
# The 90 repeated groups (same PARID + DEEDBOOK + DEEDPAGE + SALEDATE, 01 §2.5):
# what differs, is one row coded 99, and can both rows be valid-coded?
show("""
WITH g AS (SELECT PARID, DEEDBOOK, DEEDPAGE, SALEDATE, count(*) AS n,
                  count(DISTINCT SALECODE) AS codes, count(DISTINCT PRICE) AS prices,
                  count(DISTINCT INSTRTYP) AS instruments, count(DISTINCT RECORDDATE) AS record_dates,
                  count(*) FILTER (WHERE SALECODE = '99') AS rows_99,
                  count(*) FILTER (WHERE SALECODE IN ('0', 'U', 'UR')) AS valid_rows
           FROM s GROUP BY PARID, DEEDBOOK, DEEDPAGE, SALEDATE HAVING count(*) > 1)
SELECT count(*) AS groups, sum(n) AS rows,
       count(*) FILTER (WHERE codes > 1) AS code_differs,
       count(*) FILTER (WHERE prices > 1) AS price_differs,
       count(*) FILTER (WHERE instruments > 1) AS instrument_differs,
       count(*) FILTER (WHERE record_dates > 1) AS record_date_differs,
       count(*) FILTER (WHERE rows_99 >= 1) AS has_a_99_row,
       count(*) FILTER (WHERE rows_99 = 2) AS both_99,
       count(*) FILTER (WHERE valid_rows = 1) AS one_valid_row,
       count(*) FILTER (WHERE valid_rows >= 2) AS two_valid_rows
FROM g
""", width=300)

# The same counts for the 88 groups of 01 §2.4 (same parcel + sale date, one deed)
show("""
WITH g AS (SELECT PARID, SALEDATE,
                  count(DISTINCT SALECODE) AS codes, count(DISTINCT PRICE) AS prices,
                  count(DISTINCT INSTRTYP) AS instruments, count(DISTINCT RECORDDATE) AS record_dates,
                  count(*) FILTER (WHERE SALECODE = '99') AS rows_99
           FROM s GROUP BY PARID, SALEDATE
           HAVING count(*) > 1 AND count(DISTINCT (DEEDBOOK, DEEDPAGE)) = 1)
SELECT count(*) AS groups,
       count(*) FILTER (WHERE codes > 1) AS code_differs,
       count(*) FILTER (WHERE prices > 1) AS price_differs,
       count(*) FILTER (WHERE instruments > 1) AS instrument_differs,
       count(*) FILTER (WHERE record_dates > 1) AS record_date_differs,
       count(*) FILTER (WHERE rows_99 >= 1) AS has_a_99_row
FROM g
""", width=300)

┌────────┬────────┬──────────────┬───────────────┬────────────────────┬─────────────────────┬──────────────┬─────────┬───────────────┬────────────────┐
│ groups │  rows  │ code_differs │ price_differs │ instrument_differs │ record_date_differs │ has_a_99_row │ both_99 │ one_valid_row │ two_valid_rows │
│ int64  │ int128 │    int64     │     int64     │       int64        │        int64        │    int64     │  int64  │     int64     │     int64      │
├────────┼────────┼──────────────┼───────────────┼────────────────────┼─────────────────────┼──────────────┼─────────┼───────────────┼────────────────┤
│     90 │    180 │           77 │            25 │                  7 │                   3 │           61 │       0 │            23 │              0 │
└────────┴────────┴──────────────┴───────────────┴────────────────────┴─────────────────────┴──────────────┴─────────┴───────────────┴────────────────┘

┌────────┬──────────────┬───────────────┬────────────────────┬─────────────────────┬───

**What I found:** 90 groups, 180 rows. The sale code differs in 77, the price in 25, the instrument type in 7 and the record date in 3. 61 groups have a row coded 99, and none has two. 23 groups have exactly one valid-coded row, and none has two. For the 88 groups the counts are 75 / 24 / 7 / 3, with a 99 row in 59.

**So:** a valid-sale filter keeps at most one row from each group, so gold never has to choose between them. Silver still keeps both rows and flags them; which row is right stays open.

**Question C:** what share of each year file is valid-coded, and what share hasn't been reviewed yet (AA)? And when the source corrects a sale code, which way does it change?

In [23]:
# Share of valid-coded and not-yet-analysed (AA) rows in each year file
show("""
SELECT file_year, count(*) AS rows,
       count(*) FILTER (WHERE SALECODE IN ('0', 'U', 'UR')) AS valid,
       count(*) FILTER (WHERE SALECODE = 'AA') AS undetermined,
       round(100.0 * count(*) FILTER (WHERE SALECODE IN ('0', 'U', 'UR')) / count(*), 1) AS pct_valid,
       round(100.0 * count(*) FILTER (WHERE SALECODE = 'AA') / count(*), 1) AS pct_undetermined
FROM s GROUP BY 1 ORDER BY 1
""")

# When the source corrected a record (same key removed and added back, 01 §2.5), how did its sale code change?
diffs = [t for (t,) in con.execute("SELECT diff_table FROM meta.change_log ORDER BY detected_at").fetchall()]
show("SELECT from_code, to_code, count(*) AS records FROM (" + " UNION ALL ".join(
    f"SELECT r.SALECODE AS from_code, a.SALECODE AS to_code FROM {t} r JOIN {t} a USING (PARID, DEEDBOOK, DEEDPAGE, SALEDATE) "
    f"WHERE r._change = 'removed' AND a._change = 'added' AND r.SALECODE <> a.SALECODE"
    for t in diffs) + ") GROUP BY ALL ORDER BY records DESC")

┌───────────┬───────┬───────┬──────────────┬───────────┬──────────────────┐
│ file_year │ rows  │ valid │ undetermined │ pct_valid │ pct_undetermined │
│   int32   │ int64 │ int64 │    int64     │  double   │      double      │
├───────────┼───────┼───────┼──────────────┼───────────┼──────────────────┤
│      2012 │ 33651 │  2607 │         9903 │       7.7 │             29.4 │
│      2013 │ 34095 │  9199 │          882 │      27.0 │              2.6 │
│      2014 │ 33670 │  5615 │          120 │      16.7 │              0.4 │
│      2015 │ 34686 │  5828 │          108 │      16.8 │              0.3 │
│      2016 │ 36011 │  7684 │           40 │      21.3 │              0.1 │
│      2017 │ 35282 │  8105 │          474 │      23.0 │              1.3 │
│      2018 │ 36488 │  6834 │         1999 │      18.7 │              5.5 │
│      2019 │ 35671 │  4578 │         1665 │      12.8 │              4.7 │
│      2020 │ 35348 │  8226 │            6 │      23.3 │              0.0 │
│      2021 

**What I found:** the valid-coded share is 7.7% in 2012, 27.0% in 2013, between 12.8% (2019) and 23.3% (2020) in 2014–2021, 30.2–32.2% in 2022–2025, and 29.9% in 2026. The AA share is 29.4% in 2012, 5.5% in 2018, 4.7% in 2019, 5.2% in 2026, and 0.0% in 2020–2025.

**So:** the number of valid-coded rows per year follows how the County coded that year, so it can't be read as market activity. The current year's valid count keeps rising as its AA rows get reviewed. So gold shows the AA share next to every number and marks the current year as provisional. 2012 has the lowest coverage of all years.

**Question D:** two questions about deed references. (1) Are page `00002` and `2` the same page? If removing leading zeros merges references, do the merged ones look like one sale (same sale date, same parcels)? (2) For real references with more than one sale date: one parcel or several, inside one file or across files, and how far apart are the dates?

In [24]:
# Would removing leading zeros (page 00002 -> 2) merge real deed references?
show("""
WITH r AS (SELECT DISTINCT DEEDBOOK, DEEDPAGE,
                  ltrim(trim(DEEDBOOK), '0') AS book, ltrim(trim(DEEDPAGE), '0') AS page
           FROM s WHERE trim(DEEDBOOK, '0 ') <> '' AND trim(DEEDPAGE, '0 ') <> '')
SELECT count(*) AS real_references, count(DISTINCT (book, page)) AS after_removing_zeros,
       count(*) - count(DISTINCT (book, page)) AS merged
FROM r
""")

# The references that would merge: one transfer written two ways, or different transfers?
show("""
WITH r AS (SELECT *, ltrim(trim(DEEDBOOK), '0') AS book, ltrim(trim(DEEDPAGE), '0') AS page
           FROM s WHERE trim(DEEDBOOK, '0 ') <> '' AND trim(DEEDPAGE, '0 ') <> '')
SELECT book, page, list(DISTINCT DEEDBOOK || '/' || DEEDPAGE) AS written_as,
       count(DISTINCT SALEDATE) AS sale_dates, count(DISTINCT PARID) AS parcels,
       count(DISTINCT MUNICODE) AS municipalities
FROM r GROUP BY 1, 2 HAVING count(DISTINCT (DEEDBOOK, DEEDPAGE)) > 1
ORDER BY sale_dates DESC, book, page
""", width=300)

# Real deed references with more than one sale date: one parcel or several, inside one file or across files?
show("""
SELECT CASE WHEN parcels = 1 THEN 'one parcel' ELSE 'several parcels' END AS parcels_on_reference,
       CASE WHEN files > 1 THEN 'across files' ELSE 'same file' END AS span,
       count(*) AS deed_references, sum(n) AS rows,
       median(days_apart) AS median_days_apart, max(days_apart) AS max_days_apart
FROM (SELECT DEEDBOOK, DEEDPAGE, count(DISTINCT PARID) AS parcels, count(DISTINCT file_year) AS files,
             count(*) AS n, max(SALEDATE::DATE) - min(SALEDATE::DATE) AS days_apart
      FROM s WHERE trim(DEEDBOOK, '0 ') <> '' AND trim(DEEDPAGE, '0 ') <> ''
      GROUP BY 1, 2 HAVING count(DISTINCT SALEDATE) > 1)
GROUP BY ALL ORDER BY ALL
""", width=300)

┌─────────────────┬──────────────────────┬────────┐
│ real_references │ after_removing_zeros │ merged │
│      int64      │        int64         │ int64  │
├─────────────────┼──────────────────────┼────────┤
│          459199 │               459193 │      6 │
└─────────────────┴──────────────────────┴────────┘

┌─────────┬─────────┬─────────────────────────┬────────────┬─────────┬────────────────┐
│  book   │  page   │       written_as        │ sale_dates │ parcels │ municipalities │
│ varchar │ varchar │        varchar[]        │   int64    │  int64  │     int64      │
├─────────┼─────────┼─────────────────────────┼────────────┼─────────┼────────────────┤
│ TR13    │ 137     │ [TR13/137, TR13/00137]  │          2 │       2 │              2 │
│ TR17    │ 145     │ [TR17/00145, TR17/145]  │          2 │       2 │              2 │
│ TR17    │ 166     │ [TR17/166, TR17/00166]  │          2 │       2 │              2 │
│ TR17    │ 227     │ [TR17/00227, TR17/227]  │          2 │       2 │ 

**What I found:** removing leading zeros turns 459,199 real references into 459,193, so 6 merges. In 4 of them (all treasurer books, e.g. `TR13/00137` and `TR13/137`) the two forms have different sale dates and different parcels in 2 municipalities.

**So:** leading zeros are kept as written. Removing them would change only 6 references and would join different sales in at least 4 of them. *Assumption:* a book + page that appears in different years was reused or mistyped, so it isn't one document. So anything that groups rows by deed reference flags references with more than one sale date, and the price analysis only uses references with one sale date.

**Question E:** one gold observation would be one real deed reference with one parcel and one sale date, with a valid-coded row and a price, so its price is counted once. Is every such observation clear-cut (one valid row, one price)? And how many observations would each possible grouping get?

In [25]:
# Candidate gold observation: one real deed reference with one parcel and one sale date,
# carrying a valid-coded row with a price (the price is then taken once, per deed)
observations = """
    d AS (SELECT DEEDBOOK, DEEDPAGE FROM s
          WHERE trim(DEEDBOOK, '0 ') <> '' AND trim(DEEDPAGE, '0 ') <> ''
          GROUP BY 1, 2 HAVING count(DISTINCT PARID) = 1 AND count(DISTINCT SALEDATE) = 1),
    o AS (SELECT DEEDBOOK, DEEDPAGE, any_value(file_year) AS sale_year, any_value(MUNICODE) AS muni,
                 any_value(SCHOOLCODE) AS school, quarter(any_value(SALEDATE)::DATE) AS qtr,
                 count(*) AS valid_rows, count(DISTINCT PRICE) AS prices
          FROM s JOIN d USING (DEEDBOOK, DEEDPAGE)
          WHERE SALECODE IN ('0', 'U', 'UR') AND trim(PRICE) <> '' GROUP BY 1, 2)
"""

# Is each observation unambiguous (one valid row, one price)?
show(f"""
WITH {observations}
SELECT count(*) AS observations,
       count(*) FILTER (WHERE valid_rows > 1) AS with_2plus_valid_rows,
       count(*) FILTER (WHERE prices > 1) AS with_conflicting_prices
FROM o
""")

# How many observations fall in each cell of three candidate gold grains?
show(f"""
WITH {observations},
     cells AS (SELECT 'year x municipality' AS grain, count(*) AS n FROM o GROUP BY sale_year, muni
               UNION ALL SELECT 'year x school district', count(*) FROM o GROUP BY sale_year, school
               UNION ALL SELECT 'quarter x municipality', count(*) FROM o GROUP BY sale_year, qtr, muni)
SELECT grain, count(*) AS cells, quantile_disc(n, [0.1, 0.25, 0.5, 0.75]) AS n_p10_p25_p50_p75,
       count(*) FILTER (WHERE n < 10) AS cells_under_10, count(*) FILTER (WHERE n < 5) AS cells_under_5,
       sum(n) AS observations
FROM cells GROUP BY 1 ORDER BY 1
""", width=300)

# Year x municipality, per year: how many municipalities have observations, and how many cells are small?
show(f"""
WITH {observations},
     cells AS (SELECT sale_year, muni, count(*) AS n FROM o GROUP BY 1, 2)
SELECT sale_year, count(*) AS municipalities, sum(n) AS observations, median(n) AS median_per_municipality,
       count(*) FILTER (WHERE n < 10) AS municipalities_under_10
FROM cells GROUP BY 1 ORDER BY 1
""")

┌──────────────┬───────────────────────┬─────────────────────────┐
│ observations │ with_2plus_valid_rows │ with_conflicting_prices │
│    int64     │         int64         │          int64          │
├──────────────┼───────────────────────┼─────────────────────────┤
│       114432 │                     0 │                       0 │
└──────────────┴───────────────────────┴─────────────────────────┘

┌────────────────────────┬───────┬────────────────────┬────────────────┬───────────────┬──────────────┐
│         grain          │ cells │ n_p10_p25_p50_p75  │ cells_under_10 │ cells_under_5 │ observations │
│        varchar         │ int64 │      int64[]       │     int64      │     int64     │    int128    │
├────────────────────────┼───────┼────────────────────┼────────────────┼───────────────┼──────────────┤
│ quarter x municipality │  8804 │ [1, 3, 7, 16]      │           5271 │          3481 │       114432 │
│ year x municipality    │  2489 │ [4, 9, 21, 58]     │            665 │     

**What I found:** 114,432 observations, none with two valid rows or two prices. Together with the rows left out in A (8 on single-parcel deeds with several sale dates, 165 on multi-parcel deeds, 6 on placeholders), this accounts for all 114,611 valid-coded rows: 114,432 + 8 + 165 + 6 = 114,611.

Per grouping:
- **year × municipality:** 2,489 groups, median 21 observations, 665 groups under 10
- **year × school district:** 661 groups, median 124, 27 under 10
- **quarter × municipality:** 8,804 groups, median 7, 5,271 under 10

Per year, the median municipality has 8 observations in 2012, 14 in 2019, and 16 to 31 in the other years.

**So:** quarters are too thin. Year × municipality works if groups with fewer than 10 observations are flagged (about 27% of them). Year × school district is the safer alternative.

## 5. Findings and open questions
### 5.1 All findings
Every finding from notebooks 01–03 in one place. **Becomes** says what each finding turned into in the pipeline:
- **transform**: a cleaning rule or flag in silver (`pipeline/silver.py`) or a rule in gold (`pipeline/gold.py`)
- **hard / warn / info check**: a check in `pipeline/checks.py` (by name). Hard stops the build, warn is reported, info is counted on every build
- **reject**: the row goes to `silver.rejected_records` with the reason
- **accepted**: known, no action needed, with the reason

| ID | Finding | Where | Becomes |
|---|---|---|---|
| F-01 | Loaded rows equal the row count WPRDC reports for every year file. | 01 §1.1 | hard: `silver_rows_reconcile`; the source count itself is checked in bronze |
| F-02 | `_id` is a row number per file, not a sale id. It's unique inside one file but means nothing across files, and it's renumbered when the source re-publishes a file. It can't be the key. | 01 §2.2 | transform: kept as `source_row_id` (tracing only); silver's row id is `record_hash` |
| F-03 | Each row is one parcel on one deed (except blank/zero deed references), and PARID alone can't be the key. | 01 §2.3–2.4 | transform: silver keeps one row per source record |
| F-04 | The 2026 data dictionary has no descriptions; the earlier years' descriptions are assumed to apply. | 01 §0 | accepted: the earlier years' descriptions are used |
| F-05 | The pipeline's records (landing files, row counts) match what's on disk and in the tables; the run history covers 25–29 Oct 2026. | 01 §1.0 | hard: `bronze_matches_manifest` |
| F-06 | No rows from the wrong year: the files are split by sale year, and 2026 isn't finished. (The same sale can still appear in two files with different sale years: F-23.) | 01 §1.2 | warn: `sale_date_outside_file_year`; gold: `is_partial_year` |
| F-07 | The source re-publishes files with new `_id` values; past years can change (rows added and removed), and recent sale codes get corrected (400 in 3 days, 398 of them out of AA). | 01 §1.3, §2.5; 03 §4.7 C | transform: silver and gold are rebuilt from the bronze versions in use on every run |
| F-08 | Every column arrives as text; types are set in silver. | 01 §2.1 | transform: types set in silver |
| F-09 | Blank/zero deed references aren't real deeds (mostly non-deed documents); treat as unknown, keep and flag. | 01 §2.4 | transform: `is_deed_placeholder`; hard: `no_placeholder_deed_groups`; info: `placeholder_deed_rows` |
| F-10 | PARID + DEEDBOOK + DEEDPAGE + SALEDATE matches records but isn't unique: it repeats in 90 groups (180 rows; 61 with a 99 row; none with two valid-coded rows). PARID and SALEDATE are always filled; parcel details aren't part of the key; the key follows records through source corrections (2 removed keys didn't come back). | 01 §2.5; 03 §4.7 B | transform: `is_repeated_parcel_deed_date`; info: `repeated_parcel_deed_date_rows`; hard: `gold_one_valid_row_per_sale` |
| F-11 | PARID is always 16 characters with leading zeros: keep as text. | 02 §3.1 | reject: PARID not 16 digits/capitals |
| F-12 | Dates are year-month-day (not dd/mm/yyyy); RECORDDATE has blanks and one impossible year. | 02 §3.2 | transform: read as `%Y-%m-%d`; reject: not a date; hard: `reject_rate_per_year`; info: `record_date_blank`, `record_date_out_of_range` |
| F-13 | PRICE is a whole number or blank; $1, $0 and $10 are token amounts. | 02 §3.3 | transform: integer, blank → NULL; reject: not a whole number; info: `price_blank`, `price_10m_or_more` |
| F-14 | Codes keep leading zeros (stay text); MUNIDESC has extra spaces. | 02 §3.4 | transform: codes stay text, descriptions trimmed; warn: `code_format` |
| F-15 | Address: state is always PA, house number 0 = unknown, single-space fractions, extra spaces; ZIP is a postal code and crosses the county line. | 02 §3.5 | transform: trim, only spaces → NULL, `has_house_number`; warn: `code_format` (ZIP, house number) |
| F-16 | Deed book/page: placeholders, books starting with TR, page leading zeros (2 vs 00002). Kept as written, because removing zeros would merge only 6 references, mostly different sales. | 02 §3.6; 03 §4.7 D | transform: kept as written (trimmed), placeholders → NULL |
| F-17 | Blanks are mostly in early years; sale coding changed over time. | 02 §3.7 | gold: `pct_valid`, `pct_undetermined` next to every median |
| F-18 | No unusual characters; fixed-length columns confirmed. | 02 §3.8 | accepted: no character cleaning needed |
| F-19 | Recording delay (half within 6 days, 99% within 221); some sales are after their record date. | 03 §4.1 | info: `record_date_before_sale_date` |
| F-20 | The most recent months are still being recorded. | 03 §4.1 | gold: `is_partial_year`, `is_provisional` |
| F-21 | Most rows aren't market prices; adding up PRICE over rows counts money twice. | 03 §4.2 | gold: valid sale codes only, medians, never a sum of prices |
| F-22 | Multi-parcel deeds repeat the total price on each parcel (24,408 of 25,170). Treasurer deeds are the main exception (256 of 301 have different prices per parcel; 112 of the 357 deeds with mixed instrument types also differ). | 03 §4.3 | gold: only deeds with one parcel; hard: `gold_valid_rows_reconcile` |
| F-23 | No exact duplicates; 11 parcel + deed pairs are in two year files, probably a wrong year. | 03 §4.4 | info: `deed_references_in_several_years`; gold: only deeds with one sale date |
| F-24 | Each code has one description, except OTHER (21 codes, including valid U and UR); keep the code. | 03 §4.5 | transform: validity from the code lookup; warn: `code_description_not_one_to_one`, `sale_code_not_in_lookup` |
| F-25 | _Withdrawn._ The latest record date being after the latest sale date is recording delay, not a future date (0 record dates in the future, 02 §3.2). | 03 §4.6 | — |
| F-26 | The source rewrites files in place and serves half-written files under the old last_modified; such downloads are refused and tried again later. | 01 §1.3 | bronze: the download is refused and retried on the next run |
| F-27 | Valid-coded rows are almost all on single-parcel deeds: 114,440 of 114,611; 165 are on multi-parcel deeds and 6 on placeholders. None has a blank price. | 03 §4.7 A | gold: one observation = one valid single-parcel deed; rows left out are counted in `valid_records_excluded` |
| F-28 | The valid-coded share per year ranges from 7.7% (2012) to 32.2% (2023); unreviewed (AA) rows are 29.4% of 2012 and 5.2% of 2026; corrections move rows out of AA. | 03 §4.7 C | gold: `pct_undetermined`, `pct_valid`; `is_provisional` = latest sale year |
| F-29 | A deed reference with several sale dates is usually several parcels close in time within one file, but 293 references appear in more than one year file, years apart. | 03 §4.7 D | transform: `silver.deed_reference.sale_year_count`; gold: only deeds with one sale date |
| F-30 | One observation per single-parcel, single-date valid deed gives 114,432 observations: median 21 per year × municipality (665 of 2,489 groups under 10), median 124 per year × school district, median 7 per quarter × municipality. | 03 §4.7 E | gold: one row per sale year × municipality; `is_low_sample` under 10 sales, no median under 5 |
| F-31 | Some deed book and page values are typing artefacts (e.g. book `` `17274 ``, `GD-2022`, `03/08/20`; page `W`, `2/457`, `500.00`). | 01 §2.4; 02 §3.6 | accepted: kept as written; nothing uses them |
| F-32 | In 60 cases the same parcel has valid-coded rows on one sale date under two different real deeds; in 59 both deeds would pass the gold rule, so both prices would be used (46 with the same price, 14 different). | 03 §4.4 | gold: neither deed is used, both rows counted in `valid_records_excluded`; hard: `gold_one_sale_per_parcel_date` |
| F-33 | 182 valid-coded resales of the same parcel are under 180 days apart (36 under 30 days; 47 at the same price, 75 more than 20% higher). | 03 §4.4 | accepted for now: both sales are used |